# 01 — Google Cluster 2019: ограниченная локальная выгрузка

Запускайте после `sh lab bootstrap` в локальном Jupyter. При Run All по умолчанию выполняется только вывод SQL, без сети и запросов. Синтетика в ноутбуках 02 и 03 работает без Google Cloud.

Сначала настройте проект и ADC через `sh lab configure` и `sh lab auth`; см. README и docs/DATA_ACCESS.md. Затем включите CONNECT_GOOGLE для metadata/dry run. EXECUTE_QUERIES включает отдельные запросы с расходованием квоты Sandbox с ограничением бюджета.


In [ ]:
from pathlib import Path
from IPython.display import display
from vmforecast import __version__
from vmforecast.settings import Settings

SETTINGS = Settings.load()
ROOT = SETTINGS.root
LAB_DATA_DIR = SETTINGS.data
LAB_RUNS_DIR = SETTINGS.runs
print('vmforecast-lab', __version__)


In [ ]:
# All dependencies and local paths are prepared by the launcher.


In [ ]:
from vmforecast.google2019 import ExtractionConfig, BoundedExtractor, build_queries

CONNECT_GOOGLE = False
EXECUTE_QUERIES = False
PROJECT_ID = SETTINGS.project

CFG = ExtractionConfig(source_project=SETTINGS.source_project, cell='a', hours=6, n_services=20,
    max_query_bytes=5*1024**3, total_budget_bytes=15*1024**3)

DATA_DIR = LAB_DATA_DIR/'google2019_a_6h_20'

for name, sql in build_queries(CFG).items():
    print(name, sql)


## Авторизация

`sh lab auth` создаёт Application Default Credentials штатной утилитой Google Cloud CLI. Ноутбук использует их автоматически. Личные токены и ключи не нужно вставлять в ячейки или `.env`. При истёкшем/отозванном доступе повторите `sh lab auth` в терминале.

CONNECT_GOOGLE=False отключает сетевые действия. EXECUTE_QUERIES=False разрешает metadata и dry run, но не выполнение SQL-запросов с расходованием квоты Sandbox.


In [ ]:
extractor = None
if CONNECT_GOOGLE:
    if not PROJECT_ID:
        raise ValueError('Run sh lab configure --project YOUR_PROJECT_ID, then sh lab auth')
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    extractor = BoundedExtractor(client, CFG, DATA_DIR)
    meta = extractor.metadata()
    print({name: {'rows': m['num_rows'], 'bytes': m['num_bytes']} for name, m in meta.items()})
else:
    print('Google Cloud connection disabled.')


In [ ]:
if extractor is not None:
    exported = extractor.extract(execute=EXECUTE_QUERIES)
else:
    print('Пропуск выгрузки. Для демонстрации откройте notebook 02.')


## Подготовка и проверка
Строгая версия требует полного покрытия каждой задачи и отсутствия перезапусков. Ошибка оставляет quality.json с причиной. Можно сократить интервал/размер когорты, явно выбрать другие ключи или отдельно реализовать lifecycle replay.

**EXPLORATORY_COMPLETE_CASE=True** — опциональный демонстрационный отбор по полноте будущих данных. Он явно маркируется как survivorship bias и сохраняет список отброшенных задач. Не использовать такой отбор для общих выводов без отдельного анализа смещения.

Ёмкости ниже синтетические, в той же нормированной шкале, что trace. Они рассчитываются только по warm-up prefix; это не восстановление реального парка Google.


In [ ]:
import numpy as np
import pandas as pd
from vmforecast.data import prepare_usage, workload_from_tidy, save_workload
EXPLORATORY_COMPLETE_CASE = False
WARMUP = 12
if (DATA_DIR/'usage.parquet').exists() and (DATA_DIR/'events.parquet').exists():
    raw = pd.read_parquet(DATA_DIR/'usage.parquet')
    events = pd.read_parquet(DATA_DIR/'events.parquet')
    tidy, qa = prepare_usage(raw, events, CFG.t0, CFG.t1,
        report_path=DATA_DIR/'quality.json', complete_case=EXPLORATORY_COMPLETE_CASE)
    print(qa)
    tidy.to_parquet(DATA_DIR/'tidy.parquet', index=False)
    n = tidy.service_id.nunique()
    fleet_size = max(2, (n+3)//4)
    prefix = tidy[tidy.tick < WARMUP]
    max_per_service = prefix.groupby('service_id')[['cpu','mem']].max()
    # Conservative starting capacity; choose denser fleets explicitly in later experiments.
    unit_cap = np.maximum(max_per_service.max().to_numpy()*1.1,
                           max_per_service.sum().to_numpy()/fleet_size*1.5)
    caps = np.tile(np.maximum(unit_cap, 1e-8), (fleet_size,1))
    workload = workload_from_tidy(tidy, caps, warmup=WARMUP, origin_us=CFG.t0,
        provenance={'source':'Google Cluster 2019', 'cell':CFG.cell,
                    'cohort_selection':qa['cohort_selection'],
                    't0':CFG.t0,'t1':CFG.t1,'demand_proxy':'average_usage'})
    save_workload(workload, DATA_DIR/'prepared')
    print('Prepared shape:',workload.demand.shape)
else:
    print('Нет выгрузки; подготовка пропущена.')
